# Dashboard — Riesgo de Desastres Naturales en México

> **Herramienta:** Plotly (visualizaciones interactivas nativas en Jupyter)
> **Fuente:** Amazon Aurora PostgreSQL — schema `desastres`

| Visualización | Tipo | Pregunta que responde |
|---|---|---|
| 1 | Barras horizontales agrupadas | ¿Qué estados tienen más declaratorias? |
| 2 | Líneas temporales | ¿Cómo evoluciona la tendencia año a año? |
| 3 | Scatter de burbujas | ¿Hay relación entre inversión y desastres? |
| 4 | Barras + Pie | ¿Qué fenómenos son los más frecuentes? |
| 5 | Tabla con formato condicional | ¿Qué estados son foco rojo de política pública? |


## 0. Instalación de dependencias

Ejecuta solo la primera vez.

In [1]:
%pip install plotly pandas sqlalchemy psycopg2-binary --quiet

Note: you may need to restart the kernel to use updated packages.


## 1. Imports

In [12]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from sqlalchemy import create_engine, text
import plotly.io as pio          # ← AGREGAR

pio.renderers.default = "iframe" # ← AGREGAR

print('✓ Imports OK')

✓ Imports OK


## 2. Conexión a Aurora PostgreSQL

Reemplaza los valores con los de tu cluster.

In [3]:
# ─── EDITA ESTOS VALORES ──────────────────────────────────────────────────
AURORA_HOST     = 'aurora-mod4.cluster-cxjugulcqnkv.us-east-1.rds.amazonaws.com'
AURORA_PORT     = 5432
AURORA_DATABASE = 'northwind'
AURORA_USER     = 'postgres'
AURORA_PASSWORD = 'Metzme1703'
# ──────────────────────────────────────────────────────────────────────────

engine = create_engine(
    f'postgresql+psycopg2://{AURORA_USER}:{AURORA_PASSWORD}'
    f'@{AURORA_HOST}:{AURORA_PORT}/{AURORA_DATABASE}',
    pool_pre_ping=True,
)

with engine.connect() as conn:
    db = conn.execute(text('SELECT current_database()')).scalar()
    print(f'✓ Conectado a: {db}')

✓ Conectado a: northwind


## 3. Carga de datos desde Aurora

Cuatro consultas que alimentan las cinco visualizaciones.

In [4]:
# ── Vista de vulnerabilidad (CTE + NTILE — Query 5 del SQL avanzado) ──────
df_vuln = pd.read_sql(
    'SELECT * FROM desastres.v_vulnerabilidad_estados',
    engine,
)

# ── Serie temporal ────────────────────────────────────────────────────────
df_temporal = pd.read_sql(text("""
    SELECT dt.anio,
           SUM(fr.total_desastres)    AS desastres,
           SUM(fr.total_emergencias)  AS emergencias,
           SUM(fr.poblacion_afectada) AS poblacion_afectada
    FROM   desastres.fact_riesgo fr
    JOIN   desastres.dim_tiempo  dt USING (id_tiempo)
    GROUP  BY dt.anio
    ORDER  BY dt.anio
"""), engine)

# ── Distribución por fenómeno ─────────────────────────────────────────────
df_fenomeno = pd.read_sql(text("""
    SELECT df.tipo_fenomeno,
           SUM(fr.total_desastres)   AS total_desastres,
           SUM(fr.total_emergencias) AS total_emergencias,
           ROUND(SUM(fr.poblacion_afectada)::numeric, 0) AS poblacion_afectada
    FROM   desastres.fact_riesgo  fr
    JOIN   desastres.dim_fenomeno df USING (id_fenomeno)
    GROUP  BY df.tipo_fenomeno
    ORDER  BY total_desastres DESC
"""), engine)

# ── Scatter inversión vs desastres ────────────────────────────────────────
df_scatter = pd.read_sql(text("""
    SELECT de.entidad_federativa,
           SUM(fr.total_desastres)                    AS total_desastres,
           ROUND(SUM(fr.inversion_prevencion)/1e6, 2) AS inversion_millones,
           ROUND(SUM(fr.poblacion_afectada), 0)       AS poblacion_afectada
    FROM   desastres.fact_riesgo fr
    JOIN   desastres.dim_estado  de USING (id_estado)
    GROUP  BY de.entidad_federativa
    HAVING SUM(fr.total_desastres) > 0
"""), engine)

print(f'df_vuln     : {len(df_vuln):>3} filas')
print(f'df_temporal : {len(df_temporal):>3} filas')
print(f'df_fenomeno : {len(df_fenomeno):>3} filas')
print(f'df_scatter  : {len(df_scatter):>3} filas')

df_vuln     :  28 filas
df_temporal :  10 filas
df_fenomeno :   9 filas
df_scatter  :  24 filas


## 4. Paleta de colores y KPIs

In [5]:
COLOR_MAP = {
    'FOCO ROJO':                           '#C00000',
    'Alta siniestralidad':                 '#E36C09',
    'Alta siniestralidad — bien invertido': '#F6A800',
    'Siniestralidad moderada':             '#4472C4',
    'Riesgo bajo':                         '#70AD47',
}

total_des = int(df_vuln['total_desastres'].sum())
total_eme = int(df_vuln['total_emergencias'].sum())
total_pob = int(df_vuln['total_poblacion_afectada'].sum())
focos     = int((df_vuln['clasificacion_prioridad'] == 'FOCO ROJO').sum())

print('─' * 50)
print(f'  Declaratorias de desastre   : {total_des:>10,}')
print(f'  Declaratorias de emergencia : {total_eme:>10,}')
print(f'  Población afectada acumulada: {total_pob:>10,}')
print(f'  Estados foco rojo           : {focos:>10}')
print('─' * 50)

──────────────────────────────────────────────────
  Declaratorias de desastre   :        610
  Declaratorias de emergencia :        950
  Población afectada acumulada: 15,953,070
  Estados foco rojo           :          3
──────────────────────────────────────────────────


---
## Visualización 1 — Ranking de estados por siniestralidad

**Barras horizontales agrupadas:** desastres vs emergencias, top 15 estados.
Responde: *¿Qué estados concentran el mayor número de declaratorias históricas?*

In [13]:
TOP_N = 15

df_ranking = (
    df_vuln
    .sort_values('total_desastres', ascending=False)
    .head(TOP_N)
    .sort_values('total_desastres', ascending=True)
)

fig1 = px.bar(
    df_ranking,
    x=['total_desastres', 'total_emergencias'],
    y='entidad_federativa',
    orientation='h',
    title=f'Top {TOP_N} estados — Declaratorias de desastre y emergencia (acumulado histórico)',
    labels={
        'value': 'Cantidad de declaratorias',
        'entidad_federativa': '',
        'variable': 'Tipo',
    },
    color_discrete_map={
        'total_desastres':   '#1F4E79',
        'total_emergencias': '#2E75B6',
    },
    barmode='group',
    height=540,
)
fig1.update_layout(
    plot_bgcolor='white',
    legend_title_text='',
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    xaxis=dict(gridcolor='#E0E0E0'),
    font=dict(family='Arial'),
)
fig1.for_each_trace(lambda t: t.update(
    name='Desastres' if t.name == 'total_desastres' else 'Emergencias'
))
fig1.show()

---
## Visualización 2 — Tendencia temporal de eventos por año

**Líneas temporales:** evolución anual de declaratorias a nivel nacional.
Responde: *¿La frecuencia de desastres aumenta o disminuye con el tiempo?*

In [14]:
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=df_temporal['anio'],
    y=df_temporal['desastres'],
    mode='lines+markers',
    name='Desastres',
    line=dict(color='#C00000', width=2.5),
    marker=dict(size=7, symbol='circle'),
))
fig2.add_trace(go.Scatter(
    x=df_temporal['anio'],
    y=df_temporal['emergencias'],
    mode='lines+markers',
    name='Emergencias',
    line=dict(color='#F6A800', width=2.5, dash='dash'),
    marker=dict(size=7, symbol='diamond'),
))

fig2.update_layout(
    title='Declaratorias de desastre y emergencia por año — nivel nacional',
    xaxis_title='Año',
    yaxis_title='Número de declaratorias',
    plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    xaxis=dict(gridcolor='#E0E0E0', dtick=1, tickangle=-45),
    yaxis=dict(gridcolor='#E0E0E0'),
    height=420,
    font=dict(family='Arial'),
)
fig2.show()

---
## Visualización 3 — Inversión preventiva vs cantidad de desastres

**Scatter de burbujas:** cada burbuja es un estado. Tamaño = población afectada. Color = clasificación de prioridad.
Responde: *¿Los estados con más desastres reciben suficiente inversión preventiva?*

> **Cuadrante superior-izquierdo** = alta siniestralidad + baja inversión = **FOCO ROJO**.

In [15]:
df_scatter_full = df_scatter.merge(
    df_vuln[['entidad_federativa', 'clasificacion_prioridad']],
    on='entidad_federativa',
    how='left',
)

fig3 = px.scatter(
    df_scatter_full,
    x='inversion_millones',
    y='total_desastres',
    size='poblacion_afectada',
    color='clasificacion_prioridad',
    color_discrete_map=COLOR_MAP,
    text='entidad_federativa',
    title='Inversión preventiva (MXN millones) vs Total declaratorias de desastre',
    labels={
        'inversion_millones':      'Inversión preventiva (millones MXN)',
        'total_desastres':         'Total declaratorias de desastre',
        'clasificacion_prioridad': 'Clasificación',
        'poblacion_afectada':      'Población afectada',
    },
    size_max=60,
    height=580,
)
fig3.update_traces(textposition='top center', textfont_size=10)
fig3.update_layout(
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#E0E0E0'),
    yaxis=dict(gridcolor='#E0E0E0'),
    legend=dict(orientation='h', yanchor='top', y=-0.15),
    font=dict(family='Arial'),
)
fig3.show()

---
## Visualización 4 — Distribución por tipo de fenómeno

**Barras + Pie en subplots:** frecuencia de cada tipo de fenómeno natural.
Responde: *¿Qué fenómenos generan más declaratorias y cuál es su proporción?*

In [18]:
df_fen_top = df_fenomeno.dropna(subset=['tipo_fenomeno']).head(10)

# Truncar nombres muy largos para el eje Y de las barras
df_fen_top = df_fen_top.copy()
df_fen_top['fenomeno_corto'] = df_fen_top['tipo_fenomeno'].str.slice(0, 35) + \
    df_fen_top['tipo_fenomeno'].apply(lambda x: '...' if len(x) > 35 else '')

fig4 = make_subplots(
    rows=1, cols=2,
    column_widths=[0.55, 0.45],
    subplot_titles=(
        'Top 10 fenómenos — declaratorias de desastre',
        'Distribución porcentual',
    ),
    specs=[[{'type': 'bar'}, {'type': 'pie'}]],
    horizontal_spacing=0.25,   # más espacio entre los dos subplots
)

# ── Barras horizontales ────────────────────────────────────────────────────
df_fen_sorted = df_fen_top.sort_values('total_desastres', ascending=True)
fig4.add_trace(
    go.Bar(
        x=df_fen_sorted['total_desastres'],
        y=df_fen_sorted['fenomeno_corto'],
        orientation='h',
        marker_color='#1F4E79',
        showlegend=False,
        text=df_fen_sorted['total_desastres'],
        textposition='outside',
    ),
    row=1, col=1,
)

# ── Pie ───────────────────────────────────────────────────────────────────
fig4.add_trace(
    go.Pie(
        labels=df_fen_top['fenomeno_corto'],
        values=df_fen_top['total_desastres'],
        textinfo='percent',
        hole=0.35,
        marker_colors=px.colors.sequential.Blues_r[:len(df_fen_top)],
        showlegend=True,
        domain={'x': [0.58, 1.0]},   # anclar el pie a la mitad derecha
    ),
    row=1, col=2,
)

fig4.update_layout(
    height=500,
    plot_bgcolor='white',
    font=dict(family='Arial', size=12),
    margin=dict(l=280, r=20, t=60, b=40),  # margen izquierdo para las etiquetas
    legend=dict(
        orientation='v',
        x=1.02,           # leyenda fuera del gráfico a la derecha
        y=0.5,
        xanchor='left',
        font=dict(size=10),
    ),
    # Separar los títulos de los subplots
    annotations=[
        dict(text='Top 10 fenómenos — declaratorias de desastre',
             x=0.19, y=1.05, xref='paper', yref='paper',
             showarrow=False, font=dict(size=13)),
        dict(text='Distribución porcentual',
             x=0.80, y=1.05, xref='paper', yref='paper',
             showarrow=False, font=dict(size=13)),
    ],
)
fig4.update_xaxes(gridcolor='#E0E0E0', row=1, col=1)
fig4.show()

---
## Visualización 5 — Tabla de clasificación de prioridad

**Tabla con formato condicional:** resultado de la vista `v_vulnerabilidad_estados` (CTE + NTILE).
Responde: *¿Qué estados son foco rojo de política pública (alta siniestralidad + baja inversión)?*

In [17]:
cols_mostrar = [
    'entidad_federativa', 'clasificacion_prioridad',
    'total_desastres', 'total_emergencias',
    'total_poblacion_afectada', 'inversion_millones_mxn',
    'cuartil_siniestralidad', 'cuartil_baja_inversion',
]

df_tabla = (
    df_vuln[cols_mostrar]
    .copy()
    .sort_values(['cuartil_siniestralidad', 'cuartil_baja_inversion'])
    .rename(columns={
        'entidad_federativa':       'Estado',
        'clasificacion_prioridad':  'Clasificación',
        'total_desastres':          'Desastres',
        'total_emergencias':        'Emergencias',
        'total_poblacion_afectada': 'Pob. afectada',
        'inversion_millones_mxn':   'Inversión (M MXN)',
        'cuartil_siniestralidad':   'Q Siniestralidad',
        'cuartil_baja_inversion':   'Q Baja inversión',
    })
)

COLORES_CLAS = {
    'FOCO ROJO':                           'background-color:#FFCCCC; color:#C00000; font-weight:bold',
    'Alta siniestralidad':                 'background-color:#FFE5CC; color:#E36C09; font-weight:bold',
    'Alta siniestralidad — bien invertido': 'background-color:#FFF2CC; color:#7F6000',
    'Siniestralidad moderada':             'background-color:#DDEEFF; color:#1F4E79',
    'Riesgo bajo':                         'background-color:#E2EFDA; color:#375623',
}

def colorear_fila(row):
    estilo = COLORES_CLAS.get(row['Clasificación'], '')
    return [estilo if col == 'Clasificación' else '' for col in row.index]

display(
    df_tabla.style
            .apply(colorear_fila, axis=1)
            .format({
                'Pob. afectada':    '{:,.0f}',
                'Inversión (M MXN)':'{:,.2f}',
            })
            .set_table_styles([{
                'selector': 'thead th',
                'props': 'background-color:#1F4E79; color:white; font-weight:bold; padding:8px',
            }])
            .set_properties(**{'font-family': 'Arial', 'font-size': '13px'})
)

,Estado,Clasificación,Desastres,Emergencias,Pob. afectada,Inversión (M MXN),Q Siniestralidad,Q Baja inversión
0,Veracruz,FOCO ROJO,95,100,"989,730",0.00,1,1
1,Baja California Sur,FOCO ROJO,40,40,"191,500",0.00,1,1
2,Tabasco,FOCO ROJO,25,40,"933,155",0.00,1,1
3,Guerrero,Alta siniestralidad,30,35,"5,504,910",0.00,1,2
4,Durango,Alta siniestralidad,20,60,"508,230",0.00,1,2
5,Oaxaca,Alta siniestralidad,70,185,"746,565",0.00,1,3
6,Chiapas,Alta siniestralidad — bien invertido,115,140,"2,712,500",156.50,1,4
7,Sonora,Siniestralidad moderada,10,50,"356,790",0.00,2,1
8,Chihuahua,Siniestralidad moderada,10,35,"782,645",0.00,2,2
9,Colima,Siniestralidad moderada,25,10,"24,110",0.00,2,2


---
## Cierre de conexión

In [ ]:
engine.dispose()
print('✓ Conexión cerrada.')